# Tomato Variant A multiclass training notebook

This notebook trains and evaluates a 7-class tomato leaf disease classifier from the local ImageFolder data:

`data/raw/tomato/Variant-a(Multiclass Classification)/{train,val,test}/{class}/images.jpg`

It covers the full research loop:

1. repository/data setup and dataset audit
2. train/validation/test preprocessing
3. data augmentation visualization
4. demo transfer-learning architecture
5. repeatable experiment training with early stopping
6. validation and held-out test evaluation
7. plots, logs, metrics, predictions, and confusion matrices
8. checkpoint storage and AgroVision integration metadata
9. inference smoke test on the saved best model

The default experiment list contains three pretrained torchvision backbones. Edit `EXPERIMENT_CONFIGS` if you want a faster smoke run or additional architectures.


## Kaggle Setup Instructions

This notebook expects the **FarmGuard** repository to be present so it can access `configs/`, `src/`, and write outputs to `models/`, `outputs/`.

**Option 1 – Set `REPO_URL` environment variable (recommended)**

1. In the Kaggle notebook UI, click **⚙️ Settings → Environment variables**.
2. Add a variable named `REPO_URL` with the value of your FarmGuard Git repository, e.g.
   `https://github.com/your-username/FarmGuard.git` (or SSH URL if you have deploy keys).
3. The notebook will automatically clone the repo into `/kaggle/working/FarmGuard` on first run.

**Option 2 – Attach the repo as a dataset**

1. Upload the FarmGuard repository as a private dataset on Kaggle (or use an existing one).
2. In the notebook UI, **Add data → Your datasets**, select that dataset and mount it.
3. The mount path will be something like `/kaggle/input/farmguard/`. The notebook will detect it if the mounted dataset contains a `configs/` folder.

**Option 3 – Run locally**

If you run this notebook locally (VS Code, Jupyter, etc.) from the repository root, no extra steps are needed.

---

*The first code cell below will attempt to locate the repo automatically; if it fails, it will raise an error asking you to set `REPO_URL`.*

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import logging
import os
import random
import shutil
import subprocess
import sys
import traceback
import warnings
from contextlib import nullcontext

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)
    def Markdown(text):
        return text
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.utils import compute_class_weight
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, models, transforms

ROOT = Path.cwd().resolve()
# On Kaggle, the notebook may be in /kaggle/working without the repo.
# Prefer an attached repo or dataset mount, then clone the configured public repo.
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "configs").is_dir():
        ROOT = candidate
        break
else:
    for mount in sorted(Path("/kaggle/input").glob("*")):
        if (mount / "configs").is_dir():
            ROOT = mount
            break
    else:
        repo_url = os.environ.get("REPO_URL", "https://github.com/shlok-dadhich/FarmGuard.git")
        clone_root = Path("/kaggle/working/FarmGuard")
        if not clone_root.exists():
            print(f"Cloning repo from {repo_url} into {clone_root}")
            subprocess.run(["git", "clone", "--depth", "1", repo_url, str(clone_root)], check=True)
        if not (clone_root / "configs").is_dir():
            raise RuntimeError(f"Cloned repository is missing configs/: {clone_root}")
        ROOT = clone_root

sys.path.insert(0, str(ROOT))

DATA_ROOT = ROOT / "data" / "raw" / "tomato" / "Variant-a(Multiclass Classification)"
SPLIT_DIRS = {"train": "train", "val": "val", "test": "test"}
OUTPUT_ROOT = ROOT / "outputs"
METRICS_ROOT = OUTPUT_ROOT / "metrics"
LOG_ROOT = OUTPUT_ROOT / "logs"
FIGURE_ROOT = OUTPUT_ROOT / "figures" / "tomato"
CONFUSION_ROOT = OUTPUT_ROOT / "confusion_matrices"
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"
EXPERIMENT_ROOT = OUTPUT_ROOT / "experiments" / "tomato"
MODEL_ROOT = ROOT / "models" / "checkpoints" / "tomato"
CLASS_CONFIG_ROOT = ROOT / "configs" / "classes"

for directory in [OUTPUT_ROOT, METRICS_ROOT, LOG_ROOT, FIGURE_ROOT, CONFUSION_ROOT, PREDICTION_ROOT, EXPERIMENT_ROOT, MODEL_ROOT, CLASS_CONFIG_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

try:
    PROJECT_CFG = yaml.safe_load((ROOT / "configs" / "project.yaml").read_text()) or {}
except Exception:
    PROJECT_CFG = {}
try:
    TRACKING_CFG = yaml.safe_load((ROOT / "configs" / "tracking.yaml").read_text()) or {}
except Exception:
    TRACKING_CFG = {}
TRACKING_CFG["file_store"] = str(METRICS_ROOT)

SEED = int(PROJECT_CFG.get("seed", 42))
INPUT_SIZE = int(PROJECT_CFG.get("image_size", 224))
NUM_WORKERS = int(PROJECT_CFG.get("num_workers", 0))
USE_CUDA = PROJECT_CFG.get("device", "auto") in ("cuda", "gpu") and torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")

RANDOM_SEED = SEED
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
try:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
except Exception:
    pass
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOG_ROOT / "tomato_training_main.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger("tomato_training")
logger.handlers.clear()
logger.propagate = False
logger.setLevel(logging.INFO)
logger.addHandler(logging.FileHandler(LOG_ROOT / "tomato_training_main.log", encoding="utf-8"))
logger.addHandler(logging.StreamHandler())

try:
    from src.tracking.run_schema import RunRecord
    from src.tracking.tracker import get_tracker
    TRACKER = get_tracker(TRACKING_CFG)
except Exception as exc:
    logger.warning("Project tracking import failed; continuing with local files only: %s", exc)
    RunRecord = None
    TRACKER = None

VERIFY_IMAGES = True
RUN_EXPERIMENTS = True
USE_WEIGHTED_SAMPLER = True
USE_CLASS_WEIGHTS = True
LABEL_SMOOTHING = 0.05
MIN_DELTA = 0.001
EARLY_STOPPING_PATIENCE = 10
FREEZE_BACKBONE = True
UNFREEZE_EPOCH = 2
PLOT_RESULTS = True
MAX_SAMPLES_PER_CLASS = None

EXPERIMENT_CONFIGS = [
    {
        "architecture": "efficientnet_v2_s",
        "pretrained": True,
        "epochs": 25,
        "batch_size": 16,
        "lr": 1.0e-4,
        "weight_decay": 0.05,
    },
    {
        "architecture": "resnet50",
        "pretrained": True,
        "epochs": 25,
        "batch_size": 16,
        "lr": 1.0e-4,
        "weight_decay": 0.05,
    },
    {
        "architecture": "convnext_tiny",
        "pretrained": True,
        "epochs": 25,
        "batch_size": 16,
        "lr": 1.0e-4,
        "weight_decay": 0.05,
    },
    {
        "architecture": "regnet_y_4gf",
        "pretrained": True,
        "epochs": 25,
        "batch_size": 16,
        "lr": 1.0e-4,
        "weight_decay": 0.05,
    },
    {
        "architecture": "densenet121",
        "pretrained": True,
        "epochs": 25,
        "batch_size": 16,
        "lr": 1.0e-4,
        "weight_decay": 0.05,
    },
]

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def safe_name(value):
    value = str(value)
    value = value.replace(" ", "_").replace("/", "_").replace("\\", "_")
    return "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in value)

def to_jsonable(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_jsonable(v) for v in value]
    return value

def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(to_jsonable(payload), indent=2, ensure_ascii=False), encoding="utf-8")
    return path

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def relative_to_root(path):
    path = Path(path)
    try:
        return str(path.resolve().relative_to(ROOT))
    except Exception:
        return str(path.resolve())

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def total_parameters(model):
    return sum(p.numel() for p in model.parameters())

def save_checkpoint(model, path, meta):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "state_dict": model.state_dict(),
        "meta": to_jsonable(meta),
    }
    torch.save(payload, str(path))
    return str(path)

def load_checkpoint_state(path, device="cpu"):
    payload = torch.load(str(path), map_location=device, weights_only=False)
    if isinstance(payload, dict) and "state_dict" in payload:
        return payload["state_dict"], payload.get("meta", {})
    return payload, {}

def write_class_mapping(class_names):
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    save_json(MODEL_ROOT / "classes.json", {"classes": class_names, "class_to_idx": class_to_idx})
    yaml_text = "classes:\n" + "".join(f"  - {name}\n" for name in class_names)
    (CLASS_CONFIG_ROOT / "tomato.yaml").write_text(yaml_text, encoding="utf-8")
    return class_to_idx

logger.info("Repository root: %s", ROOT)
logger.info("Tomato data root: %s", DATA_ROOT)
logger.info("Device: %s", DEVICE)
display(Markdown(f"**Runtime setup complete.** Root: `{relative_to_root(ROOT)}`; device: `{DEVICE}`.`"))


In [ ]:
def has_required_splits(path):
    path = Path(path)
    return all((path / split).is_dir() for split in SPLIT_DIRS.values())

if not has_required_splits(DATA_ROOT):
    configured_data_root = os.environ.get("DATA_ROOT", "").strip()
    dataset_candidates = [
        Path(configured_data_root) if configured_data_root else None,
        ROOT / "data" / "raw" / "tomato" / "Variant-a(Multiclass Classification)",
        Path("/content/data/raw/tomato/Variant-a(Multiclass Classification)"),
        Path("/content/Variant-a(Multiclass Classification)"),
        Path("/content/drive/MyDrive/FarmGuard/data/raw/tomato/Variant-a(Multiclass Classification)"),
        Path("/kaggle/input/Variant-a(Multiclass Classification)"),
    ]
    for candidate in dataset_candidates:
        if candidate is not None and has_required_splits(candidate):
            DATA_ROOT = candidate
            break

if not has_required_splits(DATA_ROOT):
    search_roots = [Path("/content"), Path("/content/drive/MyDrive"), Path("/kaggle/input")]
    for search_root in search_roots:
        if not search_root.is_dir():
            continue
        for candidate in search_root.rglob("Variant-a(Multiclass Classification)"):
            if has_required_splits(candidate):
                DATA_ROOT = candidate
                break
        if has_required_splits(DATA_ROOT):
            break

if not has_required_splits(DATA_ROOT):
    if Path("/content").is_dir() and os.environ.get("COLAB_UPLOAD_DATA", "0") == "1":
        try:
            from google.colab import files
            import zipfile

            print(
                "Dataset not found. Upload a ZIP containing "
                "Variant-a(Multiclass Classification)/train, val, and test."
            )
            uploaded_files = files.upload()
            upload_root = Path("/content/uploaded_tomato_data")
            upload_root.mkdir(parents=True, exist_ok=True)
            for uploaded_name in uploaded_files:
                uploaded_path = Path(uploaded_name)
                if uploaded_path.suffix.lower() != ".zip":
                    continue
                with zipfile.ZipFile(uploaded_path) as archive:
                    archive.extractall(upload_root)
            for candidate in upload_root.rglob("Variant-a(Multiclass Classification)"):
                if has_required_splits(candidate):
                    DATA_ROOT = candidate
                    break
        except ImportError:
            pass

if not has_required_splits(DATA_ROOT):
    raise FileNotFoundError(
        "Tomato Variant A data was not found. In VS Code, right-click the local dataset "
        "folder and choose Colab: Upload to Colab, then rerun this cell. The uploaded "
        "folder must contain Variant-a(Multiclass Classification)/train, val, and test "
        "directories. Alternatively, mount the data and set DATA_ROOT to its path."
    )

logger.info("Resolved tomato data root: %s", DATA_ROOT)


In [ ]:
# Prefer a compatible local GPU whenever CUDA is available unless the config explicitly requests CPU.
requested_device = str(PROJECT_CFG.get("device", "auto")).strip().lower()
if requested_device not in {"auto", "cpu", "cuda", "gpu"}:
    raise ValueError(f"Unsupported device setting: {requested_device}")

cuda_available = torch.cuda.is_available()
cuda_compatible = False
if cuda_available:
    major, minor = torch.cuda.get_device_capability(0)
    cuda_compatible = major >= 7
    if not cuda_compatible:
        logger.warning(
            "CUDA device %s has compute capability %d.%d, but this PyTorch build requires >= 7.0; using CPU.",
            torch.cuda.get_device_name(0),
            major,
            minor,
        )

if requested_device == "cpu":
    USE_CUDA = False
else:
    USE_CUDA = cuda_available and cuda_compatible

DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
if USE_CUDA:
    logger.info("Using CUDA GPU: %s", torch.cuda.get_device_name(0))
else:
    logger.warning("Using CPU. torch.cuda.is_available()=%s", cuda_available)

display(Markdown(f"**Training device:** `{DEVICE}`" + (f" (`{torch.cuda.get_device_name(0)}`)" if USE_CUDA else "")))


In [ ]:
from pathlib import Path
from PIL import Image, UnidentifiedImageError

def discover_images():
    rows = []
    for split, split_dir in SPLIT_DIRS.items():
        split_path = DATA_ROOT / split_dir
        if not split_path.is_dir():
            raise FileNotFoundError(f"Missing split directory: {split_path}")
        class_dirs = sorted(
            p for p in split_path.iterdir()
            if p.is_dir() and not p.name.startswith(".")
        )
        if not class_dirs:
            raise RuntimeError(f"No class directories found in {split_path}")
        for class_dir in class_dirs:
            for image_path in sorted(p for p in class_dir.rglob("*") if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS):
                rows.append({
                    "split": split,
                    "path": str(image_path),
                    "label": class_dir.name,
                    "class_dir": class_dir.name,
                })
    return rows

rows = discover_images()
if not rows:
    raise RuntimeError("No supported images found under the selected Tomato Variant A root.")

audit_df = pd.DataFrame(rows)
original_counts = audit_df.groupby(["split", "label"]).size().unstack(fill_value=0)

bad_paths = set()
if VERIFY_IMAGES:
    for idx, row in audit_df.iterrows():
        try:
            with Image.open(row["path"]) as im:
                im.verify()
            with Image.open(row["path"]) as im:
                im.convert("RGB")
        except Exception as exc:
            bad_paths.add(str(Path(row["path"]).resolve()))
            if idx % 500 == 0:
                logger.warning("Checked %d/%d images; corrupt count=%d", idx, len(audit_df), len(bad_paths))
    if bad_paths:
        audit_df = audit_df[~audit_df["path"].apply(lambda p: str(Path(p).resolve()) in bad_paths)]
        logger.warning("Skipping %d unreadable images.", len(bad_paths))

class_names = sorted(audit_df["label"].unique().tolist())
if len(class_names) != 7:
    raise RuntimeError(f"Expected 7 classes for Variant A, found {len(class_names)}: {class_names}")
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
split_counts = audit_df.groupby("split").size().to_dict()
class_counts = audit_df.groupby("label").size().to_dict()

dataset_report = {
    "dataset": "tomato_variant_a_multiclass",
    "data_root": str(DATA_ROOT),
    "created_at": now_iso(),
    "split_counts": split_counts,
    "class_counts": class_counts,
    "classes": class_names,
    "class_to_idx": class_to_idx,
    "image_extensions": sorted(SUPPORTED_EXTENSIONS),
    "skipped_corrupt_images": len(bad_paths),
}
save_json(METRICS_ROOT / "tomato_dataset_report.json", dataset_report)
audit_df.to_csv(METRICS_ROOT / "tomato_dataset_audit.csv", index=False)

display(Markdown(f"**Dataset audit complete:** {len(audit_df)} valid images, {len(class_names)} classes."))
display(original_counts)
display(Markdown("Class order used by the model head:"))
display(pd.DataFrame({"index": range(len(class_names)), "class": class_names}))


In [ ]:
def make_train_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(INPUT_SIZE),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.1),
        transforms.RandomRotation(degrees=12),
        transforms.RandomAffine(
            degrees=0,
            translate=(0.1, 0.1),
            scale=(0.9, 1.1),
            shear=0.0,
            fill=0,
        ),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
        transforms.RandomGrayscale(p=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

def make_eval_transform():
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

train_transform = make_train_transform()
val_transform = make_eval_transform()
test_transform = make_eval_transform()

def show_transform_samples(paths, transform, title, n=4):
    paths = list(paths)[:n]
    fig, axes = plt.subplots(2, n, figsize=(4*n, 5))
    axes = axes.ravel()
    for i, path in enumerate(paths):
        img = Image.open(path).convert("RGB")
        axes[i].imshow(img)
        axes[i].set_title(f"original\n{Path(path).name}", fontsize=9)
        axes[i].axis("off")
    for j, path in enumerate(paths):
        img = Image.open(path).convert("RGB")
        tensor = transform(img)
        img_np = tensor.permute(1, 2, 0).cpu().numpy()
        img_np = img_np * IMAGENET_STD + IMAGENET_MEAN
        img_np = np.clip(img_np, 0, 1)
        ax = axes[n + j]
        ax.imshow(img_np)
        ax.set_title(f"transformed {j + 1}", fontsize=9)
        ax.axis("off")
    for ax in axes[len(paths) * 2:]:
        ax.axis("off")
    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()

sample_paths = audit_df[audit_df["split"] == "train"]["path"].sample(min(4, len(audit_df)), random_state=RANDOM_SEED).tolist()
show_transform_samples(sample_paths, train_transform, "Training augmentation preview")
show_transform_samples(sample_paths, val_transform, "Validation/test preprocessing preview")


In [ ]:
def get_weight_enum(architecture):

    enum_name = {

        "efficientnet_v2_s": "EfficientNet_V2_S_Weights",

        "resnet50": "ResNet50_Weights",

        "convnext_tiny": "ConvNeXt_Tiny_Weights",

        "densenet121": "DenseNet121_Weights",

        "regnet_y_4gf": "RegNet_Y_4GF_Weights",

    }.get(architecture)

    if enum_name is None:

        return None

    enum_obj = getattr(models, enum_name, None)

    if enum_obj is None:

        return None

    return getattr(enum_obj, "DEFAULT", None)



def replace_classifier_head(model, num_classes):

    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Linear):

        model.classifier = nn.Linear(model.classifier.in_features, num_classes)

        return model

    if hasattr(model, "classifier") and isinstance(model.classifier, nn.Sequential):

        for index in range(len(model.classifier) - 1, -1, -1):

            module = model.classifier[index]

            if isinstance(module, nn.Linear):

                model.classifier[index] = nn.Linear(module.in_features, num_classes)

                return model

        raise RuntimeError("Could not find a linear classifier head")

    if hasattr(model, "fc") and isinstance(model.fc, nn.Linear):

        model.fc = nn.Linear(model.fc.in_features, num_classes)

        return model

    if hasattr(model, "head") and isinstance(model.head, nn.Linear):

        model.head = nn.Linear(model.head.in_features, num_classes)

        return model

    raise RuntimeError("Architecture has no recognized linear classification head")



def build_torchvision_model(architecture, num_classes, pretrained):

    factory = getattr(models, architecture, None)

    if factory is None or not callable(factory):

        raise RuntimeError(f"torchvision.models has no callable {architecture}")

    weight_enum = get_weight_enum(architecture) if pretrained else None

    loaded_pretrained = False

    try:

        try:

            model = factory(weights=weight_enum)

            loaded_pretrained = weight_enum is not None

        except TypeError:

            model = factory(pretrained=weight_enum is not None)

            loaded_pretrained = weight_enum is not None

    except Exception as exc:

        if not pretrained:

            raise

        warnings.warn(

            f"Could not load pretrained weights for {architecture}; rebuilding with random weights. Reason: {exc}",

            RuntimeWarning,

        )

        try:

            model = factory(weights=None)

        except TypeError:

            model = factory(pretrained=False)

        loaded_pretrained = False

    model = replace_classifier_head(model, num_classes)

    model.farmguard_architecture = architecture

    model.farmguard_input_size = INPUT_SIZE

    model.farmguard_num_classes = num_classes

    model.farmguard_pretrained = loaded_pretrained

    return model



def build_model(architecture, num_classes, pretrained):

    return build_torchvision_model(architecture, num_classes, pretrained)



demo_arch = EXPERIMENT_CONFIGS[0]["architecture"]

demo_model = build_model(demo_arch, len(class_names), EXPERIMENT_CONFIGS[0]["pretrained"])

display(Markdown(f"**Demo architecture:** `{demo_arch}` with a 7-class classification head."))

display(pd.DataFrame({

    "total_parameters": [total_parameters(demo_model)],

    "trainable_parameters": [count_parameters(demo_model)],

    "input_size": [INPUT_SIZE],

    "num_classes": [len(class_names)],

    "pretrained_requested": [EXPERIMENT_CONFIGS[0]["pretrained"]],

    "pretrained_loaded": [bool(demo_model.farmguard_pretrained)],

}))

del demo_model


In [ ]:
def freeze_classifier_only(model):
    for parameter in model.parameters():
        parameter.requires_grad = False
    for attr in ("classifier", "fc", "head"):
        module = getattr(model, attr, None)
        if module is not None:
            for parameter in module.parameters():
                parameter.requires_grad = True
    return model

def unfreeze_all(model):
    for parameter in model.parameters():
        parameter.requires_grad = True
    return model

def build_optimizer(model, lr, weight_decay):
    trainable = [parameter for parameter in model.parameters() if parameter.requires_grad]
    if not trainable:
        raise RuntimeError("No trainable parameters found")
    return torch.optim.AdamW(trainable, lr=lr, weight_decay=weight_decay)

def build_scheduler(optimizer, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1.0e-6)

def make_class_weight_tensor(train_dataset, num_classes):
    labels = np.asarray(train_dataset.targets)
    counts = np.bincount(labels, minlength=num_classes)
    weights = compute_class_weight(
        class_weight="balanced",
        labels=np.arange(num_classes),
        class_counts=counts,
    )
    return torch.tensor(weights, dtype=torch.float)

def make_weighted_sampler(train_dataset, class_weights):
    class_weights = np.asarray(class_weights, dtype=float)
    class_counts = np.bincount(np.asarray(train_dataset.targets), minlength=len(class_weights))
    sample_weights = np.repeat(class_weights, class_counts)
    return WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_dataset),
        replacement=True,
        generator=torch.Generator().manual_seed(RANDOM_SEED),
    )

def make_loader(split, transform, batch_size, shuffle=False, sampler=None, generator=None):
    dataset = datasets.ImageFolder(str(DATA_ROOT / split), transform=transform)
    if sampler is None and shuffle:
        sampler = torch.utils.data.RandomSampler(dataset, generator=generator)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False if sampler is not None else shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        drop_last=False,
        sampler=sampler,
    )

train_dataset = datasets.ImageFolder(str(DATA_ROOT / "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(str(DATA_ROOT / "val"), transform=val_transform)
test_dataset = datasets.ImageFolder(str(DATA_ROOT / "test"), transform=test_transform)

if train_dataset.classes != class_names or val_dataset.classes != class_names or test_dataset.classes != class_names:
    raise RuntimeError("Dataset class order is inconsistent with the audited class list.")

train_labels = np.asarray(train_dataset.targets)
train_counts = np.bincount(train_labels, minlength=len(class_names))
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(class_names)),
    y=train_labels,
)
class_weight_tensor = torch.tensor(class_weights, dtype=torch.float)
train_sampler = make_weighted_sampler(train_dataset, class_weights) if USE_WEIGHTED_SAMPLER else None

train_loader = make_loader("train", train_transform, 16, shuffle=True, sampler=train_sampler, generator=torch.Generator().manual_seed(RANDOM_SEED))
val_loader = make_loader("val", val_transform, 32, shuffle=False)
test_loader = make_loader("test", test_transform, 32, shuffle=False)

display(Markdown("**DataLoaders built from the existing Variant A split directories.**"))
display(pd.DataFrame({
    "split": ["train", "val", "test"],
    "samples": [len(train_dataset), len(val_dataset), len(test_dataset)],
    "classes": [len(train_dataset.classes), len(val_dataset.classes), len(test_dataset.classes)],
}))
display(pd.DataFrame({"class": class_names, "train_count": train_counts, "class_weight": class_weights}))


In [8]:
def autocast_context(device, enabled):
    if enabled and device.type == "cuda":
        return torch.autocast(device_type="cuda", enabled=True)
    return nullcontext()

def make_grad_scaler(device):
    if device.type == "cuda":
        return torch.cuda.amp.GradScaler()
    return None

def evaluate_model(model, loader, class_names, device, split, use_amp=False, save_artifacts=True):
    model.eval()
    all_probs = []
    all_logits = []
    all_labels = []
    all_paths = []
    all_true_names = []
    all_pred_names = []
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss(reduction="sum")
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(loader):
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast_context(device, use_amp):
                logits = model(inputs)
            loss_sum += criterion(logits.cpu(), labels.cpu()).item()
            probs = torch.softmax(logits, dim=1)
            pred_indices = probs.argmax(dim=1)
            all_probs.append(probs.cpu())
            all_logits.append(logits.cpu())
            all_labels.append(labels.cpu())
            batch_paths = [
                str(path)
                for path, _ in loader.dataset.samples[
                    batch_idx * loader.batch_size : (batch_idx + 1) * loader.batch_size
                ]
            ]
            all_paths.extend(batch_paths)
            all_true_names.extend([class_names[int(label)] for label in labels.cpu().tolist()])
            all_pred_names.extend([class_names[int(idx)] for idx in pred_indices.cpu().tolist()])
    probs_np = np.concatenate(all_probs).astype(float)
    labels_np = np.concatenate(all_labels).astype(int)
    pred_indices_np = np.argmax(probs_np, axis=1)
    num_classes = len(class_names)
    metrics = {
        "accuracy": float(accuracy_score(labels_np, pred_indices_np)),
        "macro_precision": float(precision_score(labels_np, pred_indices_np, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(labels_np, pred_indices_np, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(labels_np, pred_indices_np, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels_np, pred_indices_np, average="weighted", zero_division=0)),
        "loss": float(loss_sum / max(1, len(labels_np))),
        "n": int(len(labels_np)),
        "per_class_recall": {},
    }
    per_class_recall = recall_score(labels_np, pred_indices_np, average=None, labels=list(range(num_classes)), zero_division=0)
    metrics["per_class_recall"] = {class_names[idx]: float(per_class_recall[idx]) for idx in range(num_classes)}
    report = classification_report(
        labels_np,
        pred_indices_np,
        labels=list(range(num_classes)),
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    cm = confusion_matrix(labels_np, pred_indices_np, labels=list(range(num_classes)))
    result = {
        "split": split,
        "metrics": metrics,
        "classification_report": report,
        "confusion_matrix": cm.tolist(),
        "true_labels": labels_np.tolist(),
        "predicted_indices": pred_indices_np.tolist(),
        "true_names": all_true_names,
        "predicted_names": all_pred_names,
        "paths": all_paths,
        "top_indices": np.argsort(-probs_np, axis=1)[:, :3].tolist(),
        "top_probs": probs_np[np.arange(len(probs_np)), pred_indices_np].tolist(),
    }
    if save_artifacts:
        stem = f"tomato_{safe_name(split)}_{now_iso().replace(':', '').replace('-', '')}_{np.random.randint(0, 100000):06d}"
        metrics_path = METRICS_ROOT / f"{stem}.json"
        save_json(metrics_path, result)
        result["metrics_path"] = str(metrics_path)
        cm_path = CONFUSION_ROOT / f"{stem}.png"
        fig, ax = plt.subplots(figsize=(8, 7))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_title(f"{split} confusion matrix")
        ax.set_xticks(range(num_classes))
        ax.set_yticks(range(num_classes))
        ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
        ax.set_yticklabels(class_names, fontsize=8)
        fig.colorbar(im, ax=ax)
        fig.tight_layout()
        fig.savefig(cm_path, dpi=160)
        plt.close(fig)
        result["confusion_matrix_path"] = str(cm_path)
        pred_df = pd.DataFrame({
            "path": result["paths"],
            "true_label": result["true_names"],
            "predicted_label": result["predicted_names"],
            "true_index": result["true_labels"],
            "predicted_index": result["predicted_indices"],
            "top1_probability": result["top_probs"],
            "top3_indices": result["top_indices"],
        })
        pred_path = PREDICTION_ROOT / f"{stem}_predictions.csv"
        pred_df.to_csv(pred_path, index=False)
        result["predictions_path"] = str(pred_path)
    return result

def plot_confusion_matrix(cm, class_names, title, path):
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(class_names, fontsize=8)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

def plot_training_history(history, path):
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df["epoch"], df["train_loss"], label="train loss")
    axes[0].plot(df["epoch"], df["val_loss"], label="val loss")
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].legend()
    axes[0].set_title("Loss")
    axes[1].plot(df["epoch"], df["val_accuracy"], label="accuracy")
    axes[1].plot(df["epoch"], df["val_macro_f1"], label="macro F1")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("score")
    axes[1].legend()
    axes[1].set_title("Validation metrics")
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)

def plot_sample_predictions(result, dataset, transform, class_names, path):
    paths = result["paths"]
    pred_indices = result["predicted_indices"]
    true_indices = result["true_labels"]
    n = min(12, len(paths))
    indices = np.linspace(0, len(paths) - 1, n, dtype=int)
    fig, axes = plt.subplots(3, 4, figsize=(15, 9))
    axes = axes.ravel()
    for ax, idx in zip(axes, indices):
        image = Image.open(paths[idx]).convert("RGB")
        tensor = dataset[idx][0]
        img_np = tensor.permute(1, 2, 0).cpu().numpy()
        img_np = img_np * np.asarray(IMAGENET_STD) + np.asarray(IMAGENET_MEAN)
        img_np = np.clip(img_np, 0, 1)
        ax.imshow(img_np)
        ax.set_title(f"true: {class_names[true_indices[idx]]}\npred: {class_names[pred_indices[idx]]}", fontsize=8)
        ax.axis("off")
    for ax in axes[len(indices):]:
        ax.axis("off")
    fig.suptitle("Sample predictions", y=1.02)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


In [9]:
def train_one_experiment(experiment, run_id, train_loader, val_loader, test_loader, class_names, class_weight_tensor):

    set_seed(experiment["seed"])

    architecture = experiment["architecture"]

    epochs = int(experiment["epochs"])

    batch_size = int(experiment["batch_size"])

    lr = float(experiment["lr"])

    weight_decay = float(experiment["weight_decay"])

    pretrained_requested = bool(experiment["pretrained"])



    run_root = EXPERIMENT_ROOT / run_id

    run_root.mkdir(parents=True, exist_ok=True)

    model = build_model(architecture, len(class_names), pretrained_requested)

    model = freeze_classifier_only(model)

    model.to(DEVICE)



    optimizer = build_optimizer(model, lr, weight_decay)

    scheduler = build_scheduler(optimizer, epochs)

    scaler = make_grad_scaler(DEVICE)

    use_amp = DEVICE.type == "cuda"

    loss_fn = nn.CrossEntropyLoss(

        weight=class_weight_tensor.to(DEVICE) if USE_CLASS_WEIGHTS else None,

        label_smoothing=LABEL_SMOOTHING,

    )



    best_f1 = -1.0

    best_epoch = 0

    best_state = None

    patience_counter = 0

    history = []

    start_time = datetime.now(timezone.utc)



    for epoch in range(1, epochs + 1):

        model.train()

        train_loss_sum = 0.0

        train_count = 0

        for inputs, labels in train_loader:

            inputs = inputs.to(DEVICE, non_blocking=True)

            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast_context(DEVICE, use_amp):

                logits = model(inputs)

                loss = loss_fn(logits, labels)

            if scaler is None:

                loss.backward()

                optimizer.step()

            else:

                scaler.scale(loss).backward()

                scaler.step(optimizer)

                scaler.update()

            train_loss_sum += loss.item() * labels.size(0)

            train_count += labels.size(0)



        if FREEZE_BACKBONE and epoch >= UNFREEZE_EPOCH:

            model = unfreeze_all(model)

            optimizer = build_optimizer(model, lr, weight_decay)

            scheduler = build_scheduler(optimizer, epochs)

            scaler = make_grad_scaler(DEVICE)



        scheduler.step()

        val_result = evaluate_model(

            model,

            val_loader,

            class_names,

            DEVICE,

            split="val",

            use_amp=use_amp,

            save_artifacts=False,

        )

        train_loss = train_loss_sum / max(1, train_count)

        val_metrics = val_result["metrics"]

        row = {

            "epoch": epoch,

            "train_loss": float(train_loss),

            "val_loss": float(val_metrics["loss"]),

            "val_accuracy": float(val_metrics["accuracy"]),

            "val_macro_precision": float(val_metrics["macro_precision"]),

            "val_macro_recall": float(val_metrics["macro_recall"]),

            "val_macro_f1": float(val_metrics["macro_f1"]),

            "val_weighted_f1": float(val_metrics["weighted_f1"]),

            "val_n": int(val_metrics["n"]),

        }

        history.append(row)

        pd.DataFrame(history).to_csv(run_root / "history.csv", index=False)



        if val_metrics["macro_f1"] > best_f1 + MIN_DELTA:

            best_f1 = val_metrics["macro_f1"]

            best_epoch = epoch

            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

            patience_counter = 0

        else:

            patience_counter += 1



        logger.info(

            "run=%s epoch=%d/%d train_loss=%.4f val_macro_f1=%.4f val_accuracy=%.4f",

            run_id,

            epoch,

            epochs,

            train_loss,

            val_metrics["macro_f1"],

            val_metrics["accuracy"],

        )

        if patience_counter >= EARLY_STOPPING_PATIENCE:

            logger.info("Early stopping at epoch %d", epoch)

            break



    if best_state is None:

        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    model.load_state_dict(best_state)



    final_val = evaluate_model(

        model,

        val_loader,

        class_names,

        DEVICE,

        split="val",

        use_amp=use_amp,

        save_artifacts=True,

    )

    final_test = evaluate_model(

        model,

        test_loader,

        class_names,

        DEVICE,

        split="test",

        use_amp=use_amp,

        save_artifacts=True,

    )



    checkpoint_path = MODEL_ROOT / f"tomato_{safe_name(architecture)}_seed{experiment['seed']}_{safe_name(run_id)}.pt"

    best_state_path = run_root / "best_state.pt"

    save_checkpoint(model, best_state_path, {

        "run_id": run_id,

        "stage": "best_validation_state",

        "architecture": architecture,

        "num_classes": len(class_names),

        "seed": experiment["seed"],

    })

    final_checkpoint_meta = {

        "run_id": run_id,

        "architecture": architecture,

        "crop": "tomato",

        "num_classes": len(class_names),

        "classes": class_names,

        "class_to_idx": {name: idx for idx, name in enumerate(class_names)},

        "seed": experiment["seed"],

        "input_size": INPUT_SIZE,

        "device": str(DEVICE),

        "pretrained_requested": pretrained_requested,

        "pretrained_loaded": bool(model.farmguard_pretrained),

        "epochs": epochs,

        "batch_size": batch_size,

        "lr": lr,

        "weight_decay": weight_decay,

        "augmentation_version": "tomato-notebook-aug-v1",

        "dataset_root": str(DATA_ROOT),

        "best_epoch": best_epoch,

        "best_val_macro_f1": float(best_f1),

        "best_val_accuracy": float(final_val["metrics"]["accuracy"]),

        "test_macro_f1": float(final_test["metrics"]["macro_f1"]),

        "test_accuracy": float(final_test["metrics"]["accuracy"]),

        "training_started_at": start_time.isoformat(),

        "training_finished_at": now_iso(),

    }

    save_checkpoint(model, checkpoint_path, final_checkpoint_meta)



    history_df = pd.DataFrame(history)

    summary_row = {

        "run_id": run_id,

        "status": "ok",

        "architecture": architecture,

        "seed": experiment["seed"],

        "pretrained_requested": pretrained_requested,

        "pretrained_loaded": bool(model.farmguard_pretrained),

        "epochs_attempted": len(history),

        "best_epoch": best_epoch,

        "best_val_macro_f1": float(best_f1),

        "best_val_accuracy": float(final_val["metrics"]["accuracy"]),

        "test_accuracy": float(final_test["metrics"]["accuracy"]),

        "test_macro_f1": float(final_test["metrics"]["macro_f1"]),

        "test_weighted_f1": float(final_test["metrics"]["weighted_f1"]),

        "parameter_count": count_parameters(model),

        "total_parameters": total_parameters(model),

        "train_samples": len(train_loader.dataset),

        "val_samples": len(val_loader.dataset),

        "test_samples": len(test_loader.dataset),

        "checkpoint_path": str(checkpoint_path),

        "relative_checkpoint_path": relative_to_root(checkpoint_path),

        "checkpoint_sha256": sha256_file(checkpoint_path),

        "training_time_s": (datetime.now(timezone.utc) - start_time).total_seconds(),

        "history_path": str(run_root / "history.csv"),

        "metadata_path": str(save_json(run_root / "metadata.json", final_checkpoint_meta)),

        "val_metrics_path": final_val.get("metrics_path", ""),

        "test_metrics_path": final_test.get("metrics_path", ""),

        "val_confusion_matrix_path": final_val.get("confusion_matrix_path", ""),

        "test_confusion_matrix_path": final_test.get("confusion_matrix_path", ""),

        "predictions_path": final_test.get("predictions_path", ""),

        "figure_path": str(run_root / "training_curves.png"),

        "sample_predictions_path": str(run_root / "sample_predictions.png"),

    }



    plot_training_history(history, run_root / "training_curves.png")

    plot_confusion_matrix(final_test["confusion_matrix"], class_names, "Test confusion matrix", run_root / "test_confusion_matrix.png")

    plot_sample_predictions(final_test, test_dataset, test_transform, class_names, run_root / "sample_predictions.png")



    save_json(run_root / "training_config.json", experiment)

    save_json(run_root / "final_test_metrics.json", final_test)

    save_json(run_root / "final_val_metrics.json", final_val)

    save_json(run_root / "summary.json", summary_row)



    if TRACKER is not None and RunRecord is not None:

        try:

            record = RunRecord(

                timestamp=now_iso(),

                run_id=run_id,

                crop="tomato",

                architecture=architecture,

                seed=experiment["seed"],

                dataset_version="variant-a-multiclass-v1",

                split_version="existing-split-v1",

                dataset_name="tomato_variant_a_multiclass",

                split="test",

                image_size=INPUT_SIZE,

                batch_size=batch_size,

                epochs=len(history),

                optimizer="adamw",

                learning_rate=lr,

                scheduler="cosine",

                weight_decay=weight_decay,

                augmentation_version="tomato-notebook-aug-v1",

                train_loss=float(history_df["train_loss"].iloc[-1]),

                val_loss=float(final_val["metrics"]["loss"]),

                accuracy=float(final_test["metrics"]["accuracy"]),

                macro_f1=float(final_test["metrics"]["macro_f1"]),

                weighted_f1=float(final_test["metrics"]["weighted_f1"]),

                per_class_recall=final_test["metrics"]["per_class_recall"],

                parameter_count=count_parameters(model),

                flops=0.0,

                training_time_s=summary_row["training_time_s"],

                inference_latency_ms=0.0,

                evaluation_samples=len(test_loader.dataset),

                confusion_matrix_path=final_test.get("confusion_matrix_path", ""),

                checkpoint_path=str(checkpoint_path),

                checkpoint_hash=summary_row["checkpoint_sha256"],

                device=str(DEVICE),

                kind="train",

                status="ok",

                notes=f"Best epoch {best_epoch}; test accuracy {final_test['metrics']['accuracy']:.4f}; test macro F1 {final_test['metrics']['macro_f1']:.4f}",

            )

            TRACKER.finish_run(record, artifacts=[str(checkpoint_path)])

        except Exception as exc:

            logger.warning("Could not append run to project tracker: %s", exc)



    logger.info("Completed run=%s test_accuracy=%.4f test_macro_f1=%.4f checkpoint=%s", run_id, final_test["metrics"]["accuracy"], final_test["metrics"]["macro_f1"], checkpoint_path)

    return {

        **summary_row,

        "history": history,

        "val_metrics": final_val["metrics"],

        "test_metrics": final_test["metrics"],

        "test_confusion_matrix": final_test["confusion_matrix"],

        "sample_predictions": {

            "paths": final_test["paths"],

            "true_labels": final_test["true_names"],

            "predicted_labels": final_test["predicted_names"],

            "top3_indices": final_test["top_indices"],

        },

    }


In [10]:
class _OptimizerStepAwareScheduler:
    def __init__(self, optimizer, epochs):
        self.optimizer = optimizer
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=epochs,
            eta_min=1.0e-6,
        )

    def step(self):
        if not getattr(self.optimizer, "_opt_called", False):
            return
        self.scheduler.step()


def build_scheduler(optimizer, epochs):
    return _OptimizerStepAwareScheduler(optimizer, epochs)


_original_unfreeze_all = unfreeze_all
_original_freeze_backbone = FREEZE_BACKBONE


def unfreeze_all(model):
    global FREEZE_BACKBONE
    model = _original_unfreeze_all(model)
    FREEZE_BACKBONE = False
    return model


# Restore the configured freeze behavior after each experiment so every run starts identically.
_original_train_one_experiment = train_one_experiment


def train_one_experiment(*args, **kwargs):
    global FREEZE_BACKBONE
    FREEZE_BACKBONE = _original_freeze_backbone
    try:
        return _original_train_one_experiment(*args, **kwargs)
    finally:
        FREEZE_BACKBONE = _original_freeze_backbone
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
# Kaggle/local experiment settings and torchvision compatibility fixes.
TRAINING_EPOCHS = 70
EARLY_STOPPING_PATIENCE = 25

for experiment_config in EXPERIMENT_CONFIGS:
    experiment_config["epochs"] = TRAINING_EPOCHS

# torchvision does not expose regnet_y_4gf; use the closest supported 3.2GF model
# while retaining the project-facing architecture name in checkpoints and reports.
_original_get_weight_enum = get_weight_enum


def get_weight_enum(architecture):
    if architecture == "regnet_y_4gf":
        weight_enum = getattr(models, "RegNet_Y_3_2GF_Weights", None)
        return getattr(weight_enum, "DEFAULT", None) if weight_enum is not None else None
    return _original_get_weight_enum(architecture)


if not hasattr(models, "regnet_y_4gf"):
    models.regnet_y_4gf = models.regnet_y_3_2gf

regnet_probe = build_model("regnet_y_4gf", len(class_names), pretrained=False)
assert getattr(regnet_probe, "farmguard_num_classes", None) == len(class_names)
del regnet_probe

logger.info(
    "Experiment settings: epochs=%d patience=%d device=%s root=%s",
    TRAINING_EPOCHS,
    EARLY_STOPPING_PATIENCE,
    DEVICE,
    ROOT,
)


In [ ]:
results = []
for experiment in EXPERIMENT_CONFIGS:
    if not RUN_EXPERIMENTS:
        break
    experiment["seed"] = int(experiment.get("seed", RANDOM_SEED))
    run_id = f"{safe_name(experiment['architecture'])}_seed{experiment['seed']}_{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}"
    logger.info("Starting experiment run_id=%s architecture=%s epochs=%d", run_id, experiment["architecture"], experiment["epochs"])
    try:
        result = train_one_experiment(
            experiment,
            run_id,
            train_loader,
            val_loader,
            test_loader,
            class_names,
            class_weight_tensor,
        )
        results.append(result)
        logger.info("Finished experiment run_id=%s", run_id)
    except Exception as exc:
        logger.exception("Experiment failed run_id=%s architecture=%s", run_id, experiment["architecture"])
        results.append({
            "run_id": run_id,
            "status": "failed",
            "architecture": experiment["architecture"],
            "seed": experiment["seed"],
            "error": repr(exc),
            "traceback": traceback.format_exc(),
        })

summary_df = pd.DataFrame(results)
summary_df.to_csv(METRICS_ROOT / "tomato_training_summary.csv", index=False)
save_json(EXPERIMENT_ROOT / "latest_results.json", {
    "created_at": now_iso(),
    "device": str(DEVICE),
    "results": results,
})
display(summary_df)
if not any(row.get("status") == "ok" for row in results):
    raise RuntimeError("No tomato experiment completed successfully. Check logs and GPU/disk/network availability.")


In [ ]:
successful = [row for row in results if row.get("status") == "ok"]
if successful:
    best = max(successful, key=lambda row: float(row.get("test_macro_f1", -1.0)))
    best_arch = safe_name(best["architecture"])
    alias_path = MODEL_ROOT / f"tomato_{best_arch}_best.pt"
    if alias_path.exists():
        alias_path = MODEL_ROOT / f"tomato_{best_arch}_best_{datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')}.pt"
    shutil.copy2(best["checkpoint_path"], alias_path)
    class_to_idx = write_class_mapping(class_names)
    final_summary = {
        "best_run_id": best["run_id"],
        "best_checkpoint": str(alias_path),
        "best_relative_checkpoint": relative_to_root(alias_path),
        "best_architecture": best["architecture"],
        "best_seed": best["seed"],
        "best_test_accuracy": best["test_accuracy"],
        "best_test_macro_f1": best["test_macro_f1"],
        "best_test_weighted_f1": best["test_weighted_f1"],
        "class_mapping": {"classes": class_names, "class_to_idx": class_to_idx},
        "all_results": results,
    }
    save_json(METRICS_ROOT / "tomato_training_final_summary.json", final_summary)
    display(Markdown(f"**Best run:** `{best['run_id']}` on `{best['architecture']}`. Checkpoint copied to `{relative_to_root(alias_path)}`."))
    display(pd.DataFrame([{
        "run_id": best["run_id"],
        "architecture": best["architecture"],
        "seed": best["seed"],
        "test_accuracy": best["test_accuracy"],
        "test_macro_f1": best["test_macro_f1"],
        "test_weighted_f1": best["test_weighted_f1"],
        "checkpoint": relative_to_root(alias_path),
    }]))
else:
    display(Markdown("No successful run was available for model registration."))


In [ ]:
if successful:
    best = max(successful, key=lambda row: float(row.get("test_macro_f1", -1.0)))
    best_checkpoint = Path(best["checkpoint_path"])
    if not best_checkpoint.exists():
        best_checkpoint = Path(best["relative_checkpoint_path"]) if "relative_checkpoint_path" in best else alias_path
    payload = torch.load(str(best_checkpoint), map_location="cpu", weights_only=False)
    state_dict = payload["state_dict"] if isinstance(payload, dict) and "state_dict" in payload else payload
    meta = payload.get("meta", {}) if isinstance(payload, dict) else {}
    registered_model = build_model(meta.get("architecture", best["architecture"]), len(class_names), pretrained=False)
    registered_model.load_state_dict(state_dict, strict=True)
    registered_model.to(DEVICE)
    registered_model.eval()

    example_path = Path(test_dataset.samples[0][0])
    example_image = Image.open(example_path).convert("RGB")
    with torch.no_grad():
        example_input = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(INPUT_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])(example_image).unsqueeze(0).to(DEVICE)
        start = datetime.now(timezone.utc)
        for _ in range(5):
            registered_model(example_input)
        latency_ms = (datetime.now(timezone.utc) - start).total_seconds() * 1000.0 / 5
        logits = registered_model(example_input)
        probs = torch.softmax(logits, dim=1)[0].cpu().numpy()
        top_indices = np.argsort(-probs)[:3]
        prediction_record = {
            "image": str(example_path),
            "true_label": class_names[int(test_dataset.samples[0][1])],
            "top_predictions": [
                {"label": class_names[int(idx)], "probability": float(probs[int(idx)])}
                for idx in top_indices
            ],
            "latency_ms_per_forward": float(latency_ms),
            "checkpoint": relative_to_root(best_checkpoint),
            "architecture": meta.get("architecture", best["architecture"]),
        }
    save_json(PREDICTION_ROOT / "tomato_best_example_prediction.json", prediction_record)
    display(Markdown(f"**Inference smoke test passed.** Loaded `{relative_to_root(best_checkpoint)}` and ran 5 forwards on one test image."))
    display(example_image)
    display(pd.DataFrame(prediction_record["top_predictions"]))
else:
    display(Markdown("Skipping inference smoke test because no model was trained."))


## Project integration checklist

The notebook writes the best checkpoint to:

`models/checkpoints/tomato/tomato_<architecture>_best.pt`

It also writes the class mapping to:

`configs/classes/tomato.yaml`

To expose the best model to the existing AgroVision inference/evaluation code, add one entry under `instances:` in `configs/models.yaml`:

```yaml
instances:
  - id: tomato_variant_a_best
    name: Tomato Variant A best
    architecture: efficientnet_v2_s
    crop: tomato
    checkpoint: models/checkpoints/tomato/tomato_efficientnet_v2_s_best.pt
    classes_file: configs/classes/tomato.yaml
    input_size: 224
```

Then evaluate it from the repository root:

```bash
python scripts/evaluate.py --crop tomato --model tomato_variant_a_best --checkpoint models/checkpoints/tomato/tomato_efficientnet_v2_s_best.pt
```

Replace the architecture/checkpoint filename with the values printed by the final training summary if another architecture won.
